In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import warnings
from sklearn.model_selection import learning_curve, train_test_split, LearningCurveDisplay, ShuffleSplit
from sklearn.datasets import make_classification

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from keras.models import Sequential
from keras.layers import Dense, Input, Conv2D, MaxPooling2D
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error
import seaborn as sb
import math
import os
from datetime import datetime

In [2]:
# model = Sequential()
# model.add(Input(shape=(250, 250, 3)))  # 250x250 RGB images
# model.add(Conv2D(32, 5, strides=2, activation="relu"))
# model.add(Conv2D(32, 3, activation="relu"))
# model.add(MaxPooling2D(3))

# # Can you guess what the current output shape is at this point? Probably not.
# # Let's just print it:
# for layer in model.get_config()['layers']:
#     print(str(layer))
#     print('------')

# print(str(model.get_config()['layers']))

## Functions

In [9]:
def get_model_1(n_inputs, n_outputs, loss_func='mae'):
	model = Sequential()
	model.add(Dense(20, input_dim=n_inputs, kernel_initializer='he_uniform', activation='relu'))
	model.add(Dense(5, input_dim=20, kernel_initializer='he_uniform', activation='relu'))
	model.add(Dense(n_outputs, kernel_initializer='he_uniform'))
	model.compile(loss=loss_func, optimizer='adam')
	return model

def error_report(mse, y_test, dataset_name='', metric='MSE'):
    print(f'{dataset_name} {metric} Predicted: {mse}')
    pixel_error = mse / len(y_test)
    meters_error = pixel_error / 27.4
    print(f'Mean average ({metric}) error per point -\nPixel error: {pixel_error:.2f}px;\nMeters error: {meters_error:.2f}m')

def _run_model(X_train, X_test, y_train, y_test, dataset_name, loss_func, num_features=3, num_targets=2):
    # from sklearn.preprocessing import StandardScaler
    # scaler = StandardScaler()
    # X_train_scaled = scaler.fit_transform(X_train)
    # X_test_scaled = scaler.fit_transform(X_test)
    
    #After we get the output we can use scaler.inverse_transform(X_test_scaled) to get the original scaled prediction values.
    # yhat_original_scale = scaler.inverse_transform(yhat) 
    
    # get model
    model = get_model_1(num_features, num_targets, loss_func)
    # fit the model on all data
    epochs = 50
    validation_split = 0.1
    history = model.fit(X_train, y_train, verbose=0, epochs=epochs, validation_split=validation_split)
    yhat = model.predict(X_test)
    mse = mean_squared_error(yhat, y_test)
    rmse = root_mean_squared_error(yhat, y_test) # Make sure yhat and y_test should be in original scale
    error_report(mse, y_test, dataset_name, metric='MSE')
    error_report(rmse, y_test, dataset_name, metric='RMSE')
    return history, yhat, rmse, model, epochs, validation_split

def run_trial(X=None, y=None, dataset_name=None, loss_func=None, num_features=3, num_targets=2):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    return X_test, y_test, *_run_model(X_train, X_test, y_train, y_test, dataset_name, loss_func,
                                       num_features, num_targets
                                      )

def associate_yt_yp(ytrue, ypred):
    r = {str(list(yt)):[] for yt in ytrue}
    d = {str(list(yt)):[] for yt in ytrue}
    for (yt,yp) in zip(ytrue, ypred):
        d[str(list(yt))] += [yp]
        r[str(list(yt))] += [yt-yp]
    return d, r

def draw_connecting_lines(d, x_offset, ax):
    for i, (k,yp) in enumerate(d.items()):
        yt = eval(k)
        #res = r[k]
        radius = root_mean_squared_error([yt]*len(yp), yp)
        #radius = average_euclidean_distance(np.array([yt]*len(yp)), np.array(yp)) # radius = sqrt(s / pi)
        #s = radius**2 * math.pi
        s = radius
        #ax.scatter(yt[0]+x_offset,yt[1],s=s, alpha=0.2)
        
        for _yp in yp:
            ax.plot([yt[0]+x_offset,_yp[0]+x_offset],[yt[1],_yp[1]], ls='dotted', lw=0.5, color='k', alpha=0.2)
        '''
        sb.kdeplot(
            x=np.array(yp)[:, 0]+x_offset, 
            y=np.array(yp)[:, 1],
            levels=100, thresh=.1,
            ax=ax,
            fill=True
        )
        '''

def plot_groundtruth_and_prediction_on_floorplan(title, dataPath, floorplanDir, y_test, y_predicted, figure_id, x_offset):
    plt.figure(figure_id)
    fig, ax = plt.subplots(figsize=(15,10))
    ax.set_aspect(0.1)
    ax.set_title("title")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    
    #x_offset = 200
    #x_offset = 0
    
    img_path = "{0}/{1}/{1}.png".format(dataPath, floorplanDir)
    img = plt.imread(img_path)
    ax.imshow(img)
    
    ax.plot(y_test[:, 0]+x_offset, y_test[:, 1], '*', color='red', alpha=0.4, ms=3)
    
    ax.plot(y_predicted[:, 0]+x_offset, y_predicted[:, 1], '.', color='blue', alpha=0.4, ms=2)
    return ax

In [4]:
def get_data_filename(deviceDir = ["oneplus5t", "xiaomi11lite5g"][1],
                      dataPath="../data",
                      mapDir="ict_fl1", 
                      date = "24dec24"
                     ):

    return f"{dataPath}/{mapDir}/{deviceDir}/{date}/combined_coord_raw_mag_{date}"

    

# Configuration 1

In [5]:
deviceDirs = ["oneplus5t"]#, "xiaomi11lite5g"]
dataPath="data"
mapDirs=["ict_fl1"]
dates = ["24dec24"]

result_history_filename = f'{dataPath}/ResultHistory.xlsx'
#fileNameSuffix = 'SpectralAnalysis_v4'
fileNameSuffix = 'SpectralAnalysis_v5'
data_filenames = []
filenames = []
x_offset = 0
dest_fig_dir = 'plotted_florplans'

for deviceDir in deviceDirs:
    _fn = get_data_filename(deviceDir = deviceDir,
                          dataPath=dataPath,
                          mapDir=mapDirs[0], 
                          date = dates[0]
                         )
    data_filenames.append( _fn )
    _ofn = _fn.replace("raw", fileNameSuffix ) + '.xlsx'
    filenames.append( _ofn )
print(data_filenames)
print( filenames )

['data/ict_fl1/oneplus5t/24dec24/combined_coord_raw_mag_24dec24']
['data/ict_fl1/oneplus5t/24dec24/combined_coord_SpectralAnalysis_v5_mag_24dec24.xlsx']


## Load Data 1

In [7]:
datasets = dict()
for fn in filenames:
    datasets[fn] = pd.read_excel(fn)

## Train and Test Models

In [ ]:
#print(fn, dataset_name)
targets = ['Cx','Cy']
loss_funcs = ['mse']#,'mae']
result_history_df = pd.read_excel(result_history_filename)

fig_id = 0
dir_path = datetime.now().strftime('%d-%b-%Y_%H%M%S')
dir_path = f'{dest_fig_dir}/{dir_path}'
if not os.path.exists(dir_path):
    os.makedirs(dir_path)
    
for filename, dataset in datasets.items():
    features = list(dataset.columns[1:-2].to_numpy())
    dataset_name = os.path.basename(filename)
    X, y = dataset[features].to_numpy(), dataset[targets].to_numpy()
    
    res = []
    for loss_func in loss_funcs:
        X_test, y_test, hist, y_pred, rmse, m, epochs, validation_split = run_trial(X, y, dataset_name=dataset_name, 
                                                          loss_func=loss_func, 
                                                          num_features=len(features), 
                                                          num_targets=len(targets)
                                                         )
        m.save(f'{data_filenames}.keras')
        fig_id += 1
        plt.figure(fig_id)
        fig, ax = plt.subplots()
        ax.plot(hist.history['loss'])
        ax.plot(hist.history['val_loss'])
        ax.set_title(f'{dataset_name} {loss_func}')
        path_comp = os.path.dirname(filename).split('/')
        fig_filename = f'{fileNameSuffix}_{path_comp[-3]}_{path_comp[-2]}_{path_comp[-1]}_{loss_func}'
        fig.savefig( f'{dir_path}/Loss_{fig_filename}.png', dpi=300)
        mfn = f'model_{fig_filename}.keras'
        # export model... 
        m.save(mfn)
        
        res += [{
            'dataset':filename,
            'loss_func':loss_func,
            'rmse':rmse,
            'epochs':epochs,
            'validation_split':validation_split,
            'model':str(m.get_config()['layers']),
            'saved_model_filepath':mfn
        }]
        # d, r = associate_yt_yp(y_test, y_pred)
        
        # for z in range(10):
        #     #print(f'Magnetic:: GT: ({X_test[z,0]},{X_test[z,1]},{X_test[z,2]})')
        #     print(f'Position:: GT: ({y_test[z,0]},{y_test[z,1]}) ; Predicted: ({y_pred[z,0]},{y_pred[z,1]})')
        # fig_id += 1
        # ax = plot_groundtruth_and_prediction_on_floorplan(loss_func, dataPath=dataPath, 
        #                                              floorplanDir=mapDirs[0], y_test=y_test, 
        #                                              y_predicted=y_pred, figure_id=fig_id, x_offset=x_offset)
        #draw_connecting_lines(d, x_offset, ax)
        plt.show()
        # path_comp = os.path.dirname(filename).split('/')
        # fig_filename = f'{fileNameSuffix}_{path_comp[-3]}_{path_comp[-2]}_{path_comp[-1]}_{loss_func}'
        
        # fig.savefig( f'{dir_path}/Loss_{fig_filename}', dpi=300)
        # ax.get_figure().savefig( f'{dir_path}/Floorplan_{fig_filename}', dpi=300)
        #break
    dfr = pd.DataFrame.from_records(res)
    
    result_history_df = pd.concat([result_history_df, dfr], ignore_index=True)
    #break
    display(dfr)

#result_history_df.to_excel(result_history_filename, index=False)  

In [ ]:
result_history_df.to_excel(result_history_filename, index=False)